In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print("✅ GPU OK")

GPU: Tesla T4
VRAM: 14.6 GB
✅ GPU OK


In [ ]:
!pip install -q peft==0.17.0 trl==0.9.6
print("✅ peft et trl mis à jour")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.9/503.9 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.8/245.8 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 10.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatibl

In [ ]:
from google.colab import files
print("Sélectionne kubernetes_pairs.jsonl...")
uploaded = files.upload()
print(f"✅ Fichier uploadé: {list(uploaded.keys())}")

Sélectionne kubernetes_pairs.jsonl...


Saving kubernetes_pairs.jsonl to kubernetes_pairs.jsonl
✅ Fichier uploadé: ['kubernetes_pairs.jsonl']


In [ ]:
import json, yaml
from datasets import Dataset

pairs = []
with open('kubernetes_pairs.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            pairs.append(json.loads(line))
        except:
            pass

print(f"✅ {len(pairs)} paires chargées")

def format_pair(example):
    output = example['output']
    if isinstance(output, dict):
        try:
            output = yaml.dump(output, default_flow_style=False, allow_unicode=True)
        except:
            output = str(output)
    elif not isinstance(output, str):
        output = str(output)
    inp = example.get('input', 'Cluster Kubernetes standard')
    if isinstance(inp, dict):
        inp = str(inp)
    return {"text": f"### Instruction:\n{example['instruction']}\n\n### Input:\n{inp}\n\n### Response:\n{output}"}

formatted = [format_pair(p) for p in pairs]
dataset = Dataset.from_list(formatted)
dataset = dataset.train_test_split(test_size=0.1, seed=42)
print(f"✅ Train: {len(dataset['train'])} | Test: {len(dataset['test'])}")
print(dataset['train'][0]['text'][:300])

✅ 112 paires chargées
✅ Train: 100 | Test: 12
### Instruction:
Déployer MongoDB avec PersistentVolume 10Gi

### Input:
{'namespace': 'default', 'mongodb_version': 'latest', 'pv_size': '10Gi'}

### Response:
apiVersion: v1
kind: PersistentVolume
metadata:
  name: mongodb-pv
spec:
  accessModes:
  - ReadWriteOnce
  capacity:
    storage: 10Gi
  l


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = "microsoft/phi-2"

print("🔄 Chargement tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("🔄 Chargement phi-2 float16 (3-5 min)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False

# LoRA sans bitsandbytes
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "dense"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("✅ Modèle prêt!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


🔄 Chargement tokenizer...
🔄 Chargement phi-2 float16 (3-5 min)...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 5,242,880 || all params: 2,784,926,720 || trainable%: 0.1882591725788749
✅ Modèle prêt!


In [ ]:
import sacrebleu

def generate_yaml(model, tokenizer, instruction, max_new_tokens=200):
    prompt = f"### Instruction:\n{instruction}\n\n### Input:\nCluster Kubernetes standard\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "### Response:" in generated:
        generated = generated.split("### Response:")[-1].strip()
    return generated

print("📊 Mesure BLEU AVANT fine-tuning (phi-2 de base)...")
test_samples = dataset['test'].select(range(min(5, len(dataset['test']))))
references = []
hypotheses_before = []

for i, sample in enumerate(test_samples):
    parts = sample['text'].split("### Response:")
    if len(parts) >= 2:
        instruction = parts[0].replace("### Instruction:", "").replace("### Input:", "").strip().split("\n")[0].strip()
        reference = parts[1].strip()
        generated = generate_yaml(model, tokenizer, instruction)
        references.append([reference])
        hypotheses_before.append(generated)
        print(f"  [{i+1}/5] ✅")

bleu = sacrebleu.corpus_bleu(hypotheses_before, [[r[0] for r in references]])
BLEU_BEFORE = bleu.score
print(f"\n📊 BLEU AVANT : {BLEU_BEFORE:.2f}")
print("   → Ce score sera comparé avec le score APRÈS pour prouver l'amélioration")

📊 Mesure BLEU AVANT fine-tuning (phi-2 de base)...
  [1/5] ✅
  [2/5] ✅
  [3/5] ✅
  [4/5] ✅
  [5/5] ✅

📊 BLEU AVANT : 2.95
   → Ce score sera comparé avec le score APRÈS pour prouver l'amélioration


In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="./kubellm-phi2-lora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_steps=50,
    warmup_steps=10,
    lr_scheduler_type="cosine",
    report_to="none",
    optim="adamw_torch",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset['train'],
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args,
)

print("🚀 Fine-tuning KubeLLM démarré...")
print(f"   {len(dataset['train'])} exemples · 3 epochs · LoRA rank=8")
print("   Durée estimée : 20-40 min sur T4\n")
trainer.train()
print("\n✅ Fine-tuning terminé!")

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
import subprocess, sys, os

packages = [
    "numpy==1.26.4",
    "transformers==4.44.0",
    "accelerate==0.34.0",
    "peft==0.12.0",
    "trl==0.9.6",
    "sacrebleu",
    "datasets",
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
    print(f"✅ {pkg}")

subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "bitsandbytes"])
print("\n✅ Installation terminée — redémarrage...")
os.kill(os.getpid(), 9)

✅ numpy==1.26.4
✅ transformers==4.44.0
✅ accelerate==0.34.0
✅ peft==0.12.0
✅ trl==0.9.6
✅ sacrebleu
✅ datasets


In [ ]:
import subprocess, sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "numpy==1.26.4",
    "accelerate==0.34.0",
    "transformers==4.38.2",
    "peft==0.17.0",
    "trl==0.9.6",
    "sacrebleu",
    "datasets"
])

subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "bitsandbytes"])

print("✅ Installation terminée — redémarrage automatique...")
import os
os.kill(os.getpid(), 9)  # Force restart kernel

In [ ]:
from google.colab import files
uploaded = files.upload()
print(f"✅ {list(uploaded.keys())}")

Saving kubernetes_pairs.jsonl to kubernetes_pairs.jsonl
✅ ['kubernetes_pairs.jsonl']


In [ ]:
import json, yaml, torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer
import sacrebleu

# ── Dataset ──
pairs = []
with open('kubernetes_pairs.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line: continue
        try: pairs.append(json.loads(line))
        except: pass

def format_pair(example):
    output = example['output']
    if isinstance(output, dict):
        try: output = yaml.dump(output, default_flow_style=False, allow_unicode=True)
        except: output = str(output)
    elif not isinstance(output, str):
        output = str(output)
    inp = example.get('input', 'Cluster Kubernetes standard')
    if isinstance(inp, dict): inp = str(inp)
    return {"text": f"### Instruction:\n{example['instruction']}\n\n### Input:\n{inp}\n\n### Response:\n{output}"}

formatted = [format_pair(p) for p in pairs]
dataset = Dataset.from_list(formatted).train_test_split(test_size=0.1, seed=42)
print(f"✅ Dataset: {len(dataset['train'])} train / {len(dataset['test'])} test")

# ── Modèle ──
MODEL_NAME = "microsoft/phi-2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("🔄 Chargement phi-2...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16,
    device_map="auto", trust_remote_code=True,
)
model.config.use_cache = False

lora_config = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["q_proj","k_proj","v_proj","dense"],
    lora_dropout=0.05, bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("✅ Modèle prêt!")

# ── BLEU avant ──
def generate_yaml(instruction, max_new_tokens=150):
    prompt = f"### Instruction:\n{instruction}\n\n### Input:\nCluster Kubernetes standard\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             temperature=0.1, do_sample=True,
                             pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return text.split("### Response:")[-1].strip() if "### Response:" in text else text

print("\n📊 BLEU AVANT fine-tuning...")
test_samples = dataset['test'].select(range(min(5, len(dataset['test']))))
references, hyp_before = [], []
for i, s in enumerate(test_samples):
    parts = s['text'].split("### Response:")
    if len(parts) >= 2:
        instruction = parts[0].replace("### Instruction:","").replace("### Input:","").strip().split("\n")[0].strip()
        reference = parts[1].strip()
        generated = generate_yaml(instruction)
        references.append([reference])
        hyp_before.append(generated)
        print(f"  [{i+1}/5] ✅")

BLEU_BEFORE = sacrebleu.corpus_bleu(hyp_before, [[r[0] for r in references]]).score
print(f"📊 BLEU AVANT : {BLEU_BEFORE:.2f}")

# ── Fine-tuning ──
training_args = TrainingArguments(
    output_dir="./kubellm-phi2-lora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_steps=50,
    warmup_steps=10,
    lr_scheduler_type="cosine",
    report_to="none",
    optim="adamw_torch",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset['train'],
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args,
)

print("\n🚀 Fine-tuning KubeLLM (20-40 min)...")
trainer.train()
print("✅ Fine-tuning terminé!")

# ── BLEU après ──
print("\n📊 BLEU APRÈS fine-tuning...")
hyp_after = []
for i, s in enumerate(test_samples):
    parts = s['text'].split("### Response:")
    if len(parts) >= 2:
        instruction = parts[0].replace("### Instruction:","").replace("### Input:","").strip().split("\n")[0].strip()
        hyp_after.append(generate_yaml(instruction))
        print(f"  [{i+1}/5] ✅")

BLEU_AFTER = sacrebleu.corpus_bleu(hyp_after, [[r[0] for r in references]]).score
improvement = ((BLEU_AFTER - BLEU_BEFORE) / max(BLEU_BEFORE, 0.1)) * 100

print(f"\n{'='*50}")
print(f"RÉSULTATS — Contribution C1")
print(f"{'='*50}")
print(f"BLEU phi-2 base : {BLEU_BEFORE:.2f}")
print(f"BLEU KubeLLM    : {BLEU_AFTER:.2f}")
print(f"Amélioration    : +{improvement:.1f}%")
print(f"Dataset         : {len(pairs)} paires")
print(f"{'='*50}")

# ── Sauvegarder ──
import json as json_module
SAVE_PATH = "./kubellm-v1"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
with open(f"{SAVE_PATH}/metrics.json", "w") as f:
    json_module.dump({
        "bleu_before": round(BLEU_BEFORE, 2),
        "bleu_after": round(BLEU_AFTER, 2),
        "improvement_pct": round(improvement, 1),
        "dataset_size": len(pairs),
        "model": "phi-2 + LoRA rank=8"
    }, f, indent=2)

!zip -r kubellm-v1.zip kubellm-v1/
from google.colab import files
files.download("kubellm-v1.zip")
print("✅ KubeLLM téléchargé!")

ImportError: cannot import name 'clear_device_cache' from 'accelerate.utils.memory' (/usr/local/lib/python3.12/dist-packages/accelerate/utils/memory.py)

In [ ]:
import numpy, accelerate, peft, trl, transformers
print(f"numpy:        {numpy.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"accelerate:   {accelerate.__version__}")
print(f"peft:         {peft.__version__}")
print(f"trl:          {trl.__version__}")
print("✅ OK")

numpy:        1.26.4
transformers: 4.44.0
accelerate:   0.34.0
peft:         0.12.0
trl:          0.9.6
✅ OK


In [ ]:
from google.colab import files
print("Sélectionne kubernetes_pairs.jsonl...")
uploaded = files.upload()
print(f"✅ {list(uploaded.keys())}")

Sélectionne kubernetes_pairs.jsonl...


Saving kubernetes_pairs.jsonl to kubernetes_pairs.jsonl
✅ ['kubernetes_pairs.jsonl']


In [ ]:
import json, yaml, torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer
import sacrebleu

# ── Dataset ──
pairs = []
with open('kubernetes_pairs.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line: continue
        try: pairs.append(json.loads(line))
        except: pass

def format_pair(example):
    output = example['output']
    if isinstance(output, dict):
        try: output = yaml.dump(output, default_flow_style=False, allow_unicode=True)
        except: output = str(output)
    elif not isinstance(output, str):
        output = str(output)
    inp = example.get('input', 'Cluster Kubernetes standard')
    if isinstance(inp, dict): inp = str(inp)
    return {"text": f"### Instruction:\n{example['instruction']}\n\n### Input:\n{inp}\n\n### Response:\n{output}"}

dataset = Dataset.from_list([format_pair(p) for p in pairs]).train_test_split(test_size=0.1, seed=42)
print(f"✅ Dataset: {len(dataset['train'])} train / {len(dataset['test'])} test")

# ── Modèle ──
MODEL_NAME = "microsoft/phi-2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("🔄 Chargement phi-2 (depuis cache — rapide)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16,
    device_map="auto", trust_remote_code=True,
)
model.config.use_cache = False

model = get_peft_model(model, LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["q_proj","k_proj","v_proj","dense"],
    lora_dropout=0.05, bias="none",
    task_type=TaskType.CAUSAL_LM,
))
model.print_trainable_parameters()
print("✅ Modèle prêt!")

# ── Fonction génération ──
def generate_yaml(instruction, max_new_tokens=150):
    prompt = f"### Instruction:\n{instruction}\n\n### Input:\nCluster Kubernetes standard\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             temperature=0.1, do_sample=True,
                             pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return text.split("### Response:")[-1].strip() if "### Response:" in text else text

# ── BLEU AVANT ──
print("\n📊 BLEU AVANT fine-tuning...")
test_samples = dataset['test'].select(range(min(5, len(dataset['test']))))
references, hyp_before = [], []
for i, s in enumerate(test_samples):
    parts = s['text'].split("### Response:")
    if len(parts) >= 2:
        instruction = parts[0].replace("### Instruction:","").replace("### Input:","").strip().split("\n")[0].strip()
        references.append([parts[1].strip()])
        hyp_before.append(generate_yaml(instruction))
        print(f"  [{i+1}/5] ✅")

BLEU_BEFORE = sacrebleu.corpus_bleu(hyp_before, [[r[0] for r in references]]).score
print(f"📊 BLEU AVANT : {BLEU_BEFORE:.2f}")

# ── FINE-TUNING ──
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset['train'],
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=TrainingArguments(
        output_dir="./kubellm-phi2-lora",
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        save_steps=50,
        warmup_steps=10,
        lr_scheduler_type="cosine",
        report_to="none",
        optim="adamw_torch",
    ),
)

print("\n🚀 Fine-tuning KubeLLM (20-40 min)...")
trainer.train()
print("✅ Fine-tuning terminé!")

# ── BLEU APRÈS ──
print("\n📊 BLEU APRÈS fine-tuning...")
hyp_after = []
for i, s in enumerate(test_samples):
    parts = s['text'].split("### Response:")
    if len(parts) >= 2:
        instruction = parts[0].replace("### Instruction:","").replace("### Input:","").strip().split("\n")[0].strip()
        hyp_after.append(generate_yaml(instruction))
        print(f"  [{i+1}/5] ✅")

BLEU_AFTER = sacrebleu.corpus_bleu(hyp_after, [[r[0] for r in references]]).score
improvement = ((BLEU_AFTER - BLEU_BEFORE) / max(BLEU_BEFORE, 0.1)) * 100

print(f"\n{'='*50}")
print(f"RÉSULTATS — Contribution C1")
print(f"{'='*50}")
print(f"BLEU phi-2 base : {BLEU_BEFORE:.2f}")
print(f"BLEU KubeLLM    : {BLEU_AFTER:.2f}")
print(f"Amélioration    : +{improvement:.1f}%")
print(f"Dataset         : {len(pairs)} paires")
print(f"{'='*50}")

# ── SAUVEGARDE ──
SAVE_PATH = "./kubellm-v1"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

with open(f"{SAVE_PATH}/metrics.json", "w") as f:
    json.dump({
        "model": "KubeLLM-v1",
        "base_model": "microsoft/phi-2",
        "method": "LoRA rank=8",
        "dataset_size": len(pairs),
        "epochs": 3,
        "bleu_before": round(BLEU_BEFORE, 2),
        "bleu_after": round(BLEU_AFTER, 2),
        "improvement_pct": round(improvement, 1),
    }, f, indent=2)

!zip -r kubellm-v1.zip kubellm-v1/
from google.colab import files
files.download("kubellm-v1.zip")
print("✅ KubeLLM téléchargé!")

ModuleNotFoundError: No module named 'trl'

In [ ]:
!pip install -q trl==0.9.6 sacrebleu datasets
print("✅ trl installé")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.8/245.8 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 11.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.

In [ ]:
import json, yaml, torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer
import sacrebleu

# ── Dataset ──
pairs = []
with open('kubernetes_pairs.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line: continue
        try: pairs.append(json.loads(line))
        except: pass

def format_pair(example):
    output = example['output']
    if isinstance(output, dict):
        try: output = yaml.dump(output, default_flow_style=False, allow_unicode=True)
        except: output = str(output)
    elif not isinstance(output, str):
        output = str(output)
    inp = example.get('input', 'Cluster Kubernetes standard')
    if isinstance(inp, dict): inp = str(inp)
    return {"text": f"### Instruction:\n{example['instruction']}\n\n### Input:\n{inp}\n\n### Response:\n{output}"}

dataset = Dataset.from_list([format_pair(p) for p in pairs]).train_test_split(test_size=0.1, seed=42)
print(f"✅ Dataset: {len(dataset['train'])} train / {len(dataset['test'])} test")

# ── Modèle ──
MODEL_NAME = "microsoft/phi-2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("🔄 Chargement phi-2 (depuis cache — rapide)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16,
    device_map="auto", trust_remote_code=True,
)
model.config.use_cache = False

model = get_peft_model(model, LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["q_proj","k_proj","v_proj","dense"],
    lora_dropout=0.05, bias="none",
    task_type=TaskType.CAUSAL_LM,
))
model.print_trainable_parameters()
print("✅ Modèle prêt!")

# ── Fonction génération ──
def generate_yaml(instruction, max_new_tokens=150):
    prompt = f"### Instruction:\n{instruction}\n\n### Input:\nCluster Kubernetes standard\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             temperature=0.1, do_sample=True,
                             pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return text.split("### Response:")[-1].strip() if "### Response:" in text else text

# ── BLEU AVANT ──
print("\n📊 BLEU AVANT fine-tuning...")
test_samples = dataset['test'].select(range(min(5, len(dataset['test']))))
references, hyp_before = [], []
for i, s in enumerate(test_samples):
    parts = s['text'].split("### Response:")
    if len(parts) >= 2:
        instruction = parts[0].replace("### Instruction:","").replace("### Input:","").strip().split("\n")[0].strip()
        references.append([parts[1].strip()])
        hyp_before.append(generate_yaml(instruction))
        print(f"  [{i+1}/5] ✅")

BLEU_BEFORE = sacrebleu.corpus_bleu(hyp_before, [[r[0] for r in references]]).score
print(f"📊 BLEU AVANT : {BLEU_BEFORE:.2f}")

# ── FINE-TUNING ──
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset['train'],
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=TrainingArguments(
        output_dir="./kubellm-phi2-lora",
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        save_steps=50,
        warmup_steps=10,
        lr_scheduler_type="cosine",
        report_to="none",
        optim="adamw_torch",
    ),
)

print("\n🚀 Fine-tuning KubeLLM (20-40 min)...")
trainer.train()
print("✅ Fine-tuning terminé!")

# ── BLEU APRÈS ──
print("\n📊 BLEU APRÈS fine-tuning...")
hyp_after = []
for i, s in enumerate(test_samples):
    parts = s['text'].split("### Response:")
    if len(parts) >= 2:
        instruction = parts[0].replace("### Instruction:","").replace("### Input:","").strip().split("\n")[0].strip()
        hyp_after.append(generate_yaml(instruction))
        print(f"  [{i+1}/5] ✅")

BLEU_AFTER = sacrebleu.corpus_bleu(hyp_after, [[r[0] for r in references]]).score
improvement = ((BLEU_AFTER - BLEU_BEFORE) / max(BLEU_BEFORE, 0.1)) * 100

print(f"\n{'='*50}")
print(f"RÉSULTATS — Contribution C1")
print(f"{'='*50}")
print(f"BLEU phi-2 base : {BLEU_BEFORE:.2f}")
print(f"BLEU KubeLLM    : {BLEU_AFTER:.2f}")
print(f"Amélioration    : +{improvement:.1f}%")
print(f"Dataset         : {len(pairs)} paires")
print(f"{'='*50}")

# ── SAUVEGARDE ──
SAVE_PATH = "./kubellm-v1"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

with open(f"{SAVE_PATH}/metrics.json", "w") as f:
    json.dump({
        "model": "KubeLLM-v1",
        "base_model": "microsoft/phi-2",
        "method": "LoRA rank=8",
        "dataset_size": len(pairs),
        "epochs": 3,
        "bleu_before": round(BLEU_BEFORE, 2),
        "bleu_after": round(BLEU_AFTER, 2),
        "improvement_pct": round(improvement, 1),
    }, f, indent=2)

!zip -r kubellm-v1.zip kubellm-v1/
from google.colab import files
files.download("kubellm-v1.zip")
print("✅ KubeLLM téléchargé!")

✅ Dataset: 100 train / 12 test


config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.34k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/1.08k [00:00<?, ?B/s]

... (5 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported

In [ ]:
!pip install -q torchao==0.16.0
!pip install -q peft==0.12.0
print("✅ torchao et peft mis à jour")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 8.7 MB/s eta 0:00:00
✅ torchao et peft mis à jour


In [ ]:
import json, yaml, torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer
import sacrebleu

# ── Dataset ──
pairs = []
with open('kubernetes_pairs.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line: continue
        try: pairs.append(json.loads(line))
        except: pass

def format_pair(example):
    output = example['output']
    if isinstance(output, dict):
        try: output = yaml.dump(output, default_flow_style=False, allow_unicode=True)
        except: output = str(output)
    elif not isinstance(output, str):
        output = str(output)
    inp = example.get('input', 'Cluster Kubernetes standard')
    if isinstance(inp, dict): inp = str(inp)
    return {"text": f"### Instruction:\n{example['instruction']}\n\n### Input:\n{inp}\n\n### Response:\n{output}"}

dataset = Dataset.from_list([format_pair(p) for p in pairs]).train_test_split(test_size=0.1, seed=42)
print(f"✅ Dataset: {len(dataset['train'])} train / {len(dataset['test'])} test")

# ── Modèle ──
MODEL_NAME = "microsoft/phi-2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("🔄 Chargement phi-2 (depuis cache — rapide)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16,
    device_map="auto", trust_remote_code=True,
)
model.config.use_cache = False

model = get_peft_model(model, LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["q_proj","k_proj","v_proj","dense"],
    lora_dropout=0.05, bias="none",
    task_type=TaskType.CAUSAL_LM,
))
model.print_trainable_parameters()
print("✅ Modèle prêt!")

# ── Fonction génération ──
def generate_yaml(instruction, max_new_tokens=150):
    prompt = f"### Instruction:\n{instruction}\n\n### Input:\nCluster Kubernetes standard\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             temperature=0.1, do_sample=True,
                             pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return text.split("### Response:")[-1].strip() if "### Response:" in text else text

# ── BLEU AVANT ──
print("\n📊 BLEU AVANT fine-tuning...")
test_samples = dataset['test'].select(range(min(5, len(dataset['test']))))
references, hyp_before = [], []
for i, s in enumerate(test_samples):
    parts = s['text'].split("### Response:")
    if len(parts) >= 2:
        instruction = parts[0].replace("### Instruction:","").replace("### Input:","").strip().split("\n")[0].strip()
        references.append([parts[1].strip()])
        hyp_before.append(generate_yaml(instruction))
        print(f"  [{i+1}/5] ✅")

BLEU_BEFORE = sacrebleu.corpus_bleu(hyp_before, [[r[0] for r in references]]).score
print(f"📊 BLEU AVANT : {BLEU_BEFORE:.2f}")

# ── FINE-TUNING ──
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset['train'],
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=TrainingArguments(
        output_dir="./kubellm-phi2-lora",
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        save_steps=50,
        warmup_steps=10,
        lr_scheduler_type="cosine",
        report_to="none",
        optim="adamw_torch",
    ),
)

print("\n🚀 Fine-tuning KubeLLM (20-40 min)...")
trainer.train()
print("✅ Fine-tuning terminé!")

# ── BLEU APRÈS ──
print("\n📊 BLEU APRÈS fine-tuning...")
hyp_after = []
for i, s in enumerate(test_samples):
    parts = s['text'].split("### Response:")
    if len(parts) >= 2:
        instruction = parts[0].replace("### Instruction:","").replace("### Input:","").strip().split("\n")[0].strip()
        hyp_after.append(generate_yaml(instruction))
        print(f"  [{i+1}/5] ✅")

BLEU_AFTER = sacrebleu.corpus_bleu(hyp_after, [[r[0] for r in references]]).score
improvement = ((BLEU_AFTER - BLEU_BEFORE) / max(BLEU_BEFORE, 0.1)) * 100

print(f"\n{'='*50}")
print(f"RÉSULTATS — Contribution C1")
print(f"{'='*50}")
print(f"BLEU phi-2 base : {BLEU_BEFORE:.2f}")
print(f"BLEU KubeLLM    : {BLEU_AFTER:.2f}")
print(f"Amélioration    : +{improvement:.1f}%")
print(f"Dataset         : {len(pairs)} paires")
print(f"{'='*50}")

# ── SAUVEGARDE ──
SAVE_PATH = "./kubellm-v1"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

with open(f"{SAVE_PATH}/metrics.json", "w") as f:
    json.dump({
        "model": "KubeLLM-v1",
        "base_model": "microsoft/phi-2",
        "method": "LoRA rank=8",
        "dataset_size": len(pairs),
        "epochs": 3,
        "bleu_before": round(BLEU_BEFORE, 2),
        "bleu_after": round(BLEU_AFTER, 2),
        "improvement_pct": round(improvement, 1),
    }, f, indent=2)

!zip -r kubellm-v1.zip kubellm-v1/
from google.colab import files
files.download("kubellm-v1.zip")
print("✅ KubeLLM téléchargé!")

✅ Dataset: 100 train / 12 test
🔄 Chargement phi-2 (depuis cache — rapide)...


Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

trainable params: 5,242,880 || all params: 2,784,926,720 || trainable%: 0.1883
✅ Modèle prêt!

📊 BLEU AVANT fine-tuning...


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer CodeGenTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


  [1/5] ✅
  [2/5] ✅
  [3/5] ✅
  [4/5] ✅
  [5/5] ✅
📊 BLEU AVANT : 1.31


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:280: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:318: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

TypeError: Trainer.__init__() got an unexpected keyword argument 'tokenizer'

In [ ]:
# ── FINE-TUNING ──
from trl import SFTConfig

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset['train'],
    args=SFTConfig(
        output_dir="./kubellm-phi2-lora",
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        save_steps=50,
        warmup_steps=10,
        lr_scheduler_type="cosine",
        report_to="none",
        optim="adamw_torch",
        dataset_text_field="text",
        max_seq_length=512,
    ),
)

print("\n🚀 Fine-tuning KubeLLM (20-40 min)...")
trainer.train()
print("✅ Fine-tuning terminé!")

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

TypeError: Trainer.__init__() got an unexpected keyword argument 'tokenizer'

In [ ]:
from trl import SFTConfig

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset['train'],
    processing_class=tokenizer,
    args=SFTConfig(
        output_dir="./kubellm-phi2-lora",
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        save_steps=50,
        warmup_steps=10,
        lr_scheduler_type="cosine",
        report_to="none",
        optim="adamw_torch",
        dataset_text_field="text",
        max_seq_length=512,
    ),
)

print("\n🚀 Fine-tuning KubeLLM (20-40 min)...")
trainer.train()
print("✅ Fine-tuning terminé!")

TypeError: SFTTrainer.__init__() got an unexpected keyword argument 'processing_class'

In [ ]:
import inspect
from trl import SFTTrainer
sig = inspect.signature(SFTTrainer.__init__)
print("Arguments SFTTrainer:")
for name, param in sig.parameters.items():
    print(f"  {name}: {param.default}")

Arguments SFTTrainer:
  self: <class 'inspect._empty'>
  model: None
  args: None
  data_collator: None
  train_dataset: None
  eval_dataset: None
  tokenizer: None
  model_init: None
  compute_metrics: None
  callbacks: None
  optimizers: (None, None)
  preprocess_logits_for_metrics: None
  peft_config: None
  dataset_text_field: None
  packing: False
  formatting_func: None
  max_seq_length: None
  infinite: None
  num_of_sequences: 1024
  chars_per_token: 3.6
  dataset_num_proc: None
  dataset_batch_size: 1000
  neftune_noise_alpha: None
  model_init_kwargs: None
  dataset_kwargs: None
  eval_packing: None


In [ ]:
from trl import SFTConfig

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset['train'],
    dataset_text_field="text",
    max_seq_length=512,
    args=SFTConfig(
        output_dir="./kubellm-phi2-lora",
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        save_steps=50,
        warmup_steps=10,
        lr_scheduler_type="cosine",
        report_to="none",
        optim="adamw_torch",
    ),
)

print("\n🚀 Fine-tuning KubeLLM (20-40 min)...")
trainer.train()
print("✅ Fine-tuning terminé!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:280: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:318: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

TypeError: Trainer.__init__() got an unexpected keyword argument 'tokenizer'

In [ ]:
!pip install -q transformers==4.40.2
import os
os.kill(os.getpid(), 9)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 67.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
sentence-transformers 5.6.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.2 which is incompatible.


In [1]:
!pip install -q trl==0.9.6 sacrebleu datasets
print("✅ OK")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.8/245.8 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 10.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompa

In [2]:
from google.colab import files
uploaded = files.upload()
print(f"✅ {list(uploaded.keys())}")

Saving kubernetes_pairs.jsonl to kubernetes_pairs.jsonl
✅ ['kubernetes_pairs.jsonl']


In [3]:
import json, yaml, torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
import sacrebleu

# Dataset
pairs = []
with open('kubernetes_pairs.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line: continue
        try: pairs.append(json.loads(line))
        except: pass

def format_pair(example):
    output = example['output']
    if isinstance(output, dict):
        try: output = yaml.dump(output, default_flow_style=False, allow_unicode=True)
        except: output = str(output)
    elif not isinstance(output, str):
        output = str(output)
    inp = example.get('input', 'Cluster Kubernetes standard')
    if isinstance(inp, dict): inp = str(inp)
    return {"text": f"### Instruction:\n{example['instruction']}\n\n### Input:\n{inp}\n\n### Response:\n{output}"}

dataset = Dataset.from_list([format_pair(p) for p in pairs]).train_test_split(test_size=0.1, seed=42)
print(f"✅ Dataset: {len(dataset['train'])} train / {len(dataset['test'])} test")

# Modèle
MODEL_NAME = "microsoft/phi-2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("🔄 Chargement phi-2...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16,
    device_map="auto", trust_remote_code=True,
)
model.config.use_cache = False
model = get_peft_model(model, LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["q_proj","k_proj","v_proj","dense"],
    lora_dropout=0.05, bias="none",
    task_type=TaskType.CAUSAL_LM,
))
model.print_trainable_parameters()
print("✅ Modèle prêt!")

# BLEU avant
def generate_yaml(instruction, max_new_tokens=150):
    prompt = f"### Instruction:\n{instruction}\n\n### Input:\nCluster Kubernetes standard\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             temperature=0.1, do_sample=True,
                             pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return text.split("### Response:")[-1].strip() if "### Response:" in text else text

print("\n📊 BLEU AVANT...")
test_samples = dataset['test'].select(range(min(5, len(dataset['test']))))
references, hyp_before = [], []
for i, s in enumerate(test_samples):
    parts = s['text'].split("### Response:")
    if len(parts) >= 2:
        instruction = parts[0].replace("### Instruction:","").replace("### Input:","").strip().split("\n")[0].strip()
        references.append([parts[1].strip()])
        hyp_before.append(generate_yaml(instruction))
        print(f"  [{i+1}/5] ✅")
BLEU_BEFORE = sacrebleu.corpus_bleu(hyp_before, [[r[0] for r in references]]).score
print(f"📊 BLEU AVANT : {BLEU_BEFORE:.2f}")

# Fine-tuning
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset['train'],
    dataset_text_field="text",
    max_seq_length=512,
    args=SFTConfig(
        output_dir="./kubellm-phi2-lora",
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        save_steps=50,
        warmup_steps=10,
        lr_scheduler_type="cosine",
        report_to="none",
        optim="adamw_torch",
    ),
)
print("\n🚀 Fine-tuning KubeLLM (20-40 min)...")
trainer.train()
print("✅ Fine-tuning terminé!")

# BLEU après
print("\n📊 BLEU APRÈS...")
hyp_after = []
for i, s in enumerate(test_samples):
    parts = s['text'].split("### Response:")
    if len(parts) >= 2:
        instruction = parts[0].replace("### Instruction:","").replace("### Input:","").strip().split("\n")[0].strip()
        hyp_after.append(generate_yaml(instruction))
        print(f"  [{i+1}/5] ✅")
BLEU_AFTER = sacrebleu.corpus_bleu(hyp_after, [[r[0] for r in references]]).score
improvement = ((BLEU_AFTER - BLEU_BEFORE) / max(BLEU_BEFORE, 0.1)) * 100

print(f"\n{'='*50}")
print(f"RÉSULTATS — Contribution C1")
print(f"{'='*50}")
print(f"BLEU phi-2 base : {BLEU_BEFORE:.2f}")
print(f"BLEU KubeLLM    : {BLEU_AFTER:.2f}")
print(f"Amélioration    : +{improvement:.1f}%")
print(f"Dataset         : {len(pairs)} paires")
print(f"{'='*50}")

# Sauvegarde
SAVE_PATH = "./kubellm-v1"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
with open(f"{SAVE_PATH}/metrics.json", "w") as f:
    json.dump({
        "model": "KubeLLM-v1",
        "base_model": "microsoft/phi-2",
        "method": "LoRA rank=8",
        "dataset_size": len(pairs),
        "epochs": 3,
        "bleu_before": round(BLEU_BEFORE, 2),
        "bleu_after": round(BLEU_AFTER, 2),
        "improvement_pct": round(improvement, 1),
    }, f, indent=2)

!zip -r kubellm-v1.zip kubellm-v1/
from google.colab import files
files.download("kubellm-v1.zip")
print("✅ KubeLLM téléchargé!")

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
import subprocess, sys, os
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "numpy==1.26.4", "--force-reinstall"])
os.kill(os.getpid(), 9)

In [1]:
!pip install -q transformers==4.40.2 peft==0.12.0 trl==0.9.6 accelerate==0.34.0 sacrebleu datasets torchao==0.16.0
!pip uninstall -y bitsandbytes -q
print("✅ Installation terminée")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 103.7 MB/s eta 0:00:00
✅ Installation terminée


In [2]:
import numpy as np
import torch
import peft
import trl
print(f"numpy: {np.__version__}")
print(f"peft:  {peft.__version__}")
print(f"trl:   {trl.__version__}")
print("✅ Tout OK")

numpy: 1.26.4
peft:  0.12.0
trl:   0.9.6
✅ Tout OK


In [3]:
from google.colab import files
uploaded = files.upload()
print(f"✅ {list(uploaded.keys())}")

Saving kubernetes_pairs.jsonl to kubernetes_pairs (1).jsonl
✅ ['kubernetes_pairs (1).jsonl']


In [4]:
import json, yaml, torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
import sacrebleu

# Dataset
pairs = []
with open('kubernetes_pairs.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line: continue
        try: pairs.append(json.loads(line))
        except: pass

def format_pair(example):
    output = example['output']
    if isinstance(output, dict):
        try: output = yaml.dump(output, default_flow_style=False, allow_unicode=True)
        except: output = str(output)
    elif not isinstance(output, str):
        output = str(output)
    inp = example.get('input', 'Cluster Kubernetes standard')
    if isinstance(inp, dict): inp = str(inp)
    return {"text": f"### Instruction:\n{example['instruction']}\n\n### Input:\n{inp}\n\n### Response:\n{output}"}

dataset = Dataset.from_list([format_pair(p) for p in pairs]).train_test_split(test_size=0.1, seed=42)
print(f"✅ Dataset: {len(dataset['train'])} train / {len(dataset['test'])} test")

# Modèle
MODEL_NAME = "microsoft/phi-2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("🔄 Chargement phi-2 (depuis cache)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16,
    device_map="auto", trust_remote_code=True,
)
model.config.use_cache = False
model = get_peft_model(model, LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["q_proj","k_proj","v_proj","dense"],
    lora_dropout=0.05, bias="none",
    task_type=TaskType.CAUSAL_LM,
))
model.print_trainable_parameters()
print("✅ Modèle prêt!")

# BLEU avant
def generate_yaml(instruction, max_new_tokens=150):
    prompt = f"### Instruction:\n{instruction}\n\n### Input:\nCluster Kubernetes standard\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             temperature=0.1, do_sample=True,
                             pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return text.split("### Response:")[-1].strip() if "### Response:" in text else text

print("\n📊 BLEU AVANT fine-tuning...")
test_samples = dataset['test'].select(range(min(5, len(dataset['test']))))
references, hyp_before = [], []
for i, s in enumerate(test_samples):
    parts = s['text'].split("### Response:")
    if len(parts) >= 2:
        instruction = parts[0].replace("### Instruction:","").replace("### Input:","").strip().split("\n")[0].strip()
        references.append([parts[1].strip()])
        hyp_before.append(generate_yaml(instruction))
        print(f"  [{i+1}/5] ✅")
BLEU_BEFORE = sacrebleu.corpus_bleu(hyp_before, [[r[0] for r in references]]).score
print(f"📊 BLEU AVANT : {BLEU_BEFORE:.2f}")

# Fine-tuning
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset['train'],
    dataset_text_field="text",
    max_seq_length=512,
    args=SFTConfig(
        output_dir="./kubellm-phi2-lora",
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        save_steps=50,
        warmup_steps=10,
        lr_scheduler_type="cosine",
        report_to="none",
        optim="adamw_torch",
    ),
)
print("\n🚀 Fine-tuning KubeLLM (20-40 min)...")
trainer.train()
print("✅ Fine-tuning terminé!")

# BLEU après
print("\n📊 BLEU APRÈS fine-tuning...")
hyp_after = []
for i, s in enumerate(test_samples):
    parts = s['text'].split("### Response:")
    if len(parts) >= 2:
        instruction = parts[0].replace("### Instruction:","").replace("### Input:","").strip().split("\n")[0].strip()
        hyp_after.append(generate_yaml(instruction))
        print(f"  [{i+1}/5] ✅")
BLEU_AFTER = sacrebleu.corpus_bleu(hyp_after, [[r[0] for r in references]]).score
improvement = ((BLEU_AFTER - BLEU_BEFORE) / max(BLEU_BEFORE, 0.1)) * 100

print(f"\n{'='*50}")
print(f"RÉSULTATS — Contribution C1")
print(f"{'='*50}")
print(f"BLEU phi-2 base : {BLEU_BEFORE:.2f}")
print(f"BLEU KubeLLM    : {BLEU_AFTER:.2f}")
print(f"Amélioration    : +{improvement:.1f}%")
print(f"Dataset         : {len(pairs)} paires")
print(f"Méthode         : LoRA rank=8, 3 epochs")
print(f"{'='*50}")

# Sauvegarde
SAVE_PATH = "./kubellm-v1"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
with open(f"{SAVE_PATH}/metrics.json", "w") as f:
    json.dump({
        "model": "KubeLLM-v1",
        "base_model": "microsoft/phi-2",
        "method": "LoRA rank=8",
        "dataset_size": len(pairs),
        "epochs": 3,
        "bleu_before": round(BLEU_BEFORE, 2),
        "bleu_after": round(BLEU_AFTER, 2),
        "improvement_pct": round(improvement, 1),
    }, f, indent=2)

!zip -r kubellm-v1.zip kubellm-v1/
from google.colab import files
files.download("kubellm-v1.zip")
print("✅ KubeLLM téléchargé!")

✅ Dataset: 100 train / 12 test


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

... (18 outputs left out)

  adding: kubellm-v1/ (stored 0%)
  adding: kubellm-v1/added_tokens.json (deflated 84%)
  adding: kubellm-v1/metrics.json (deflated 26%)
  adding: kubellm-v1/merges.txt (deflated 53%)
  adding: kubellm-v1/adapter_config.json (deflated 52%)
  adding: kubellm-v1/vocab.json (deflated 59%)
  adding: kubellm-v1/tokenizer.json (deflated 72%)
  adding: kubellm-v1/adapter_model.safetensors (deflated 8%)
  adding: kubellm-v1/special_tokens_map.json (deflated 75%)
  adding: kubellm-v1/tokenizer_config.json (deflated 94%)
  adding: kubellm-v1/README.md (deflated 66%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ KubeLLM téléchargé!
